# 🚀 GPT

In this notebook, we'll walk through the steps required to train your own GPT model on the wine review dataset

The code is adapted from the excellent [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) created by Apoorv Nandan available on the Keras website.

In [38]:
%load_ext autoreload
%autoreload 2
import numpy as np
import json
import re
import string
from IPython.display import display, HTML

import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Parameters <a name="parameters"></a>

In [2]:
VOCAB_SIZE = 10000
MAX_LEN = 80
EMBEDDING_DIM = 256
KEY_DIM = 256
N_HEADS = 2
FEED_FORWARD_DIM = 256
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 5

## 1. Load the data <a name="load"></a>

In [3]:
# Download and load the Wine Reviews dataset from Kaggle
import subprocess, sys, os

try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

# Download the dataset
dataset_path = kagglehub.dataset_download("zynicide/wine-reviews")
print(f"Dataset downloaded to: {dataset_path}")

# Load the JSON file
json_file = os.path.join(dataset_path, "winemag-data-130k-v2.json")
with open(json_file) as json_data:
    wine_data = json.load(json_data)

c:\Users\yogan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset downloaded to: C:\Users\yogan\.cache\kagglehub\datasets\zynicide\wine-reviews\versions\4


In [4]:
wine_data[10]

{'points': '87',
 'title': 'Kirkland Signature 2011 Mountain Cuvée Cabernet Sauvignon (Napa Valley)',
 'description': 'Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility.',
 'taster_name': 'Virginie Boone',
 'taster_twitter_handle': '@vboone',
 'price': 19,
 'designation': 'Mountain Cuvée',
 'variety': 'Cabernet Sauvignon',
 'region_1': 'Napa Valley',
 'region_2': 'Napa',
 'province': 'California',
 'country': 'US',
 'winery': 'Kirkland Signature'}

In [5]:
# Filter the dataset
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]

In [6]:
# Count the recipes
n_wines = len(filtered_data)
print(f"{n_wines} recipes loaded")

129907 recipes loaded


In [7]:
example = filtered_data[25]
print(example)

wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard-designated Pinot that hails from a high-elevation site. Small in production, it offers intense, full-bodied raspberry and blackberry steeped in smoky spice and smooth texture.


## 2. Tokenize the data <a name="tokenize"></a>

In [8]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [9]:
# Display an example of a recipe
example_data = text_data[25]
example_data

'wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard - designated Pinot that hails from a high - elevation site . Small in production , it offers intense , full - bodied raspberry and blackberry steeped in smoky spice and smooth texture . '

In [10]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [11]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [12]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [13]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: :
3: ,
4: .
5: and
6: the
7: wine
8: a
9: of


In [14]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[   7   10    2   20    2   29    2   43   62    2   55    5  243 4145
  453  634   26    9  497  499  667   17   12  142   14 2214   43   25
 2484   32    8  223   14 2213  948    4  594   17  987    3   15   75
  237    3   64   14   82   97    5   74 2633   17  198   49    5  125
   77    4    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0]


## 3. Create the Training Set <a name="create"></a>

In [15]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = text_ds.map(prepare_inputs)

In [16]:
example_input_output = train_ds.take(1).get_single_element()

In [17]:
# Example Input
example_input_output[0][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([   7,   10,    2,   42,    2,  254,    2,  254,   27,    2,  301,
         46,   52,  612,  455,  118,   19,  683,   54, 1371,   30,    3,
         12,  254,   13,    1,  718,   14,   58,  187,  112, 2916,    4,
         54,   38,    5,   57,  361, 8541,    6,   73,   30,    5,  647,
          3, 1368,   81,  110,    4,   35,   66,    4,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

In [18]:
# Example Output (shifted by one token)
example_input_output[1][0]

<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([  10,    2,   42,    2,  254,    2,  254,   27,    2,  301,   46,
         52,  612,  455,  118,   19,  683,   54, 1371,   30,    3,   12,
        254,   13,    1,  718,   14,   58,  187,  112, 2916,    4,   54,
         38,    5,   57,  361, 8541,    6,   73,   30,    5,  647,    3,
       1368,   81,  110,    4,   35,   66,    4,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

## 5. Create the causal attention mask function <a name="causal"></a>

In [19]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)


np.transpose(causal_attention_mask(1, 10, 10, dtype=tf.int32)[0])

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [20]:
class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return (self.ln_2(out1 + ffn_output), attention_scores)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

## 7. Create the Token and Position Embedding <a name="embedder"></a>

In [21]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [22]:
inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x, attention_scores = TransformerBlock(
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
gpt = models.Model(inputs=inputs, outputs=[outputs, attention_scores])
gpt.compile("adam", loss=[losses.SparseCategoricalCrossentropy(), None])

In [23]:
gpt.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, None, 256)      │     2,580,480 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ [(None, None, 256),    │       658,688 │
│ (TransformerBlock)              │ (None, 2, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 10000)    │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,809,168 (22.16 MB)

 Trainable params: 5,809,168 (22.16 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    gpt = models.load_model("./models/gpt", compile=True)

## 9. Train the Transformer <a name="train"></a>

In [25]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("wine review", max_tokens=80, temperature=1.0)

In [27]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [28]:
gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/5
4060/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 2.6015
generated text:
wine review : italy : tuscany : red blend : underbrush , spice and roasted coffee beans and clove take shape to the thick palate along with dried black cherry and more coffee flavors . it ' s soft and easy to enjoy now . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 400s 98ms/step - loss: 2.2465
Epoch 2/5
4060/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 1.9836
generated text:
wine review : portugal : alentejano : portuguese red : still young and fruity , this offers crisp tannins . it is a fresh wine , laced with juicy berry fruits and a soft , smoky aftertaste . delicious and likely to give it its freshness . drink from 2016 . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 430s 106ms/step - loss: 1.9571
Epoch 3/5
4060/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 1.8950
generated text:
wine review : italy : southern italy : negroamaro : subdued aromas of stone fruit , citrus blossom and mature blackberry lead the nose . th

In [31]:
# Save the final model
import os
os.makedirs("./models", exist_ok=True)
gpt.save("./models/gpt.keras")

# 3. Generate text using the Transformer

In [32]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [33]:
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=1.0
)


generated text:
wine review : us : california : cabernet sauvignon : this is the most pretty good , monumental cabernet of the vintage . it ' s big , soft and decadent , and tobacco - roasted blackberries , mocha violets , iron , beef jerky , smoked beef , cola and dark chocolate - herb palate . the tannins the wine is impressive , mixing fig , cranberry , raspberry fruit and sweet oak . 



In [34]:
info = text_generator.generate(
    "wine review : italy", max_tokens=80, temperature=0.5
)


generated text:
wine review : italy : northeastern italy : pinot grigio : this is a delicate and delicate expression of pinot grigio that shows a pretty floral fragrance of rose and yellow rose and a whiff of exotic fruit . the vibrant palate offers juicy grapefruit , lemon zest and a hint of bitter almond . 



In [35]:
info = text_generator.generate(
    "wine review : germany", max_tokens=80, temperature=0.5
)
print_probs(info, vocab)


generated text:
wine review : germany : mosel : riesling : fresh - pressed apple and pear notes abound on this off - dry riesling . it ' s delicately sweet in juicy peach and lemon flavors , with a brisk , citrusy finish . 



::   	100.0%
-:   	0.0%
grosso:   	0.0%
region:   	0.0%
&:   	0.0%
--------



mosel:   	94.94000244140625%
rheingau:   	1.9600000381469727%
rheinhessen:   	1.8799999952316284%
nahe:   	0.49000000953674316%
pfalz:   	0.3799999952316284%
--------



::   	99.69999694824219%
-:   	0.30000001192092896%
grosso:   	0.0%
valley:   	0.0%
istriana:   	0.0%
--------



riesling:   	99.98999786376953%
pinot:   	0.0%
white:   	0.0%
gewürztraminer:   	0.0%
sparkling:   	0.0%
--------



...

,:   	86.76000213623047%
acidity:   	6.389999866485596%
finish:   	1.4900000095367432%
streak:   	0.8999999761581421%
lime:   	0.699999988079071%
--------



clean:   	21.649999618530273%
steely:   	21.219999313354492%
lemony:   	17.8799991607666%
citrusy:   	13.180000305175781%
refreshing:   	7.190000057220459%
--------



finish:   	72.87999725341797%
acidity:   	20.84000015258789%
tang:   	2.069999933242798%
edge:   	1.4600000381469727%
streak:   	0.8899999856948853%
--------



.:   	99.93000030517578%
that:   	0.05999999865889549%
,:   	0.0%
with:   	0.0%
accented:   	0.0%
--------



:   	90.93000030517578%
drink:   	8.300000190734863%
it:   	0.5699999928474426%
enjoy:   	0.10000000149011612%
a:   	0.03999999910593033%
--------



In [36]:
# Generate text at temperature = 0.2 (low temperature - more deterministic/conservative)
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=0.2
)
print_probs(info, vocab)


generated text:
wine review : us : california : pinot noir : this is a very good pinot noir . it ' s dry and silky , with modest cherry , cola , persimmon and spice flavors . it ' s a little too young , so drink now . 



::   	100.0%
-:   	0.0%
grosso:   	0.0%
,:   	0.0%
region:   	0.0%
--------



california:   	99.98999786376953%
washington:   	0.009999999776482582%
oregon:   	0.0%
new:   	0.0%
virginia:   	0.0%
--------



::   	100.0%
-:   	0.0%
grosso:   	0.0%
,:   	0.0%
other:   	0.0%
--------



pinot:   	96.43000030517578%
cabernet:   	1.850000023841858%
chardonnay:   	1.590000033378601%
syrah:   	0.05999999865889549%
sauvignon:   	0.03999999910593033%
--------



...

so:   	94.04000091552734%
but:   	5.639999866485596%
and:   	0.28999999165534973%
with:   	0.029999999329447746%
although:   	0.0%
--------



drink:   	100.0%
give:   	0.0%
it:   	0.0%
the:   	0.0%
you:   	0.0%
--------



now:   	99.98999786376953%
it:   	0.0%
soon:   	0.0%
up:   	0.0%
this:   	0.0%
--------



.:   	100.0%
with:   	0.0%
,:   	0.0%
and:   	0.0%
for:   	0.0%
--------



:   	100.0%
the:   	0.0%
it:   	0.0%
could:   	0.0%
drink:   	0.0%
--------



In [37]:
# Generate text at temperature = 1.5 (high temperature - more random/creative)
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=1.5
)
print_probs(info, vocab)


generated text:
wine review : us : new york : shiraz : slightly spicy honeydew tones cast a still concentration of this full - bodied - fledged syrah ; aged tastes incredibly on and include fresh plum skin - funk and earth shot raspberries . in acidity returns in the midpalate time producing reds while previously employed two dramatically sweet oak happening within meaty muscularity ; strange among others has a bouquet notes siena imports . edward wines . 



::   	99.66000366210938%
-:   	0.029999999329447746%
grosso:   	0.009999999776482582%
,:   	0.009999999776482582%
region:   	0.009999999776482582%
--------



california:   	48.939998626708984%
washington:   	14.619999885559082%
oregon:   	10.220000267028809%
new:   	6.989999771118164%
virginia:   	1.909999966621399%
--------



york:   	89.54000091552734%
mexico:   	3.2200000286102295%
jersey:   	0.8600000143051147%
south:   	0.38999998569488525%
french:   	0.3400000035762787%
--------



::   	98.11000061035156%
-:   	0.1899999976158142%
grosso:   	0.10999999940395355%
blanc:   	0.05000000074505806%
region:   	0.03999999910593033%
--------



...

.:   	47.209999084472656%
,:   	15.850000381469727%
and:   	5.190000057220459%
;:   	1.2599999904632568%
%:   	1.2000000476837158%
--------



:   	59.66999816894531%
drink:   	4.400000095367432%
this:   	1.2200000286102295%
imported:   	0.9800000190734863%
it:   	0.9800000190734863%
--------



wines:   	48.790000915527344%
[UNK]:   	3.2899999618530273%
reds:   	1.7599999904632568%
cabernets:   	1.2699999809265137%
.:   	0.8799999952316284%
--------



.:   	59.81999969482422%
,:   	7.010000228881836%
from:   	1.690000057220459%
and:   	1.6299999952316284%
wines:   	1.4700000286102295%
--------



:   	73.37999725341797%
drink:   	2.009999990463257%
this:   	0.6899999976158142%
imported:   	0.6800000071525574%
enjoy:   	0.38999998569488525%
--------



## Interpretation: How Temperature Affects the Generated Wine Reviews

In our GPT model, the **temperature** parameter controls how "creative" or "safe" the model is when picking the next word. Think of it like a dial, turn it down and the model plays it safe, turn it up and it starts taking wild guesses.

### What We Observed

**At temperature = 0.2 (very low):**
Our model generated something like: *"this is a very good pinot noir. it's dry and silky, with modest cherry, cola, persimmon and spice flavors."*

This reads like a perfectly reasonable wine review, but it's also very predictable. The model kept choosing the most obvious next word every time. At this setting, the probability scores show 99–100% confidence on most tokens, the model isn't even considering alternatives. The downside is the text feels generic and could easily become repetitive if we generated multiple reviews.

**At temperature = 0.5:**
The model produced: *"fresh-pressed apple and pear notes abound on this off-dry riesling. it's delicately sweet in juicy peach and lemon flavors, with a brisk, citrusy finish."*

This is the sweet spot. The text is still coherent and reads naturally, but we see more interesting word choices, "fresh-pressed," "brisk," "citrusy." The model is willing to explore a bit beyond the safest option, which makes the review sound more like something a real person would write.

**At temperature = 1.0 (default):**
We got: *"this is the most pretty good, monumental cabernet of the vintage. it's big, soft and decadent, and tobacco-roasted blackberries, mocha violets..."*

Now things get more creative. Words like "monumental," "decadent," and "mocha violets" are unexpected and interesting. However, the grammar starts to slip, "most pretty good" doesn't quite make sense. The model is sampling more freely from its vocabulary, which leads to richer language but occasional awkward phrasing.

**At temperature = 1.5 (very high):**
The output was: *"slightly spicy honeydew tones cast a still concentration of this full-bodied-fledged syrah; aged tastes incredibly on and include fresh plum skin-funk and earth shot raspberries..."*

This is mostly nonsense. While a few interesting fragments appear ("spicy honeydew tones," "plum skin"), the overall text doesn't hold together. The model is essentially picking words almost at random, so the sentence structure falls apart completely.

### Why Does This Happen?

Temperature works by adjusting how the model distributes probability across its vocabulary before picking a word:

- **Low temperature (0.2)** sharpens the probabilities, the top word gets almost all the weight, so the model nearly always picks the same "best" word. It's like asking someone to write a review but telling them to only use the most common phrases.

- **Medium temperature (0.5)** keeps the distribution focused but allows some room for less common words to be picked. This creates a nice balance between making sense and sounding interesting.

- **Default temperature (1.0)** uses the probabilities as-is, without any adjustment. Less common words now have a real chance of being selected, making the output more diverse but occasionally odd.

- **High temperature (1.5)** flattens the probabilities so much that even very unlikely words become plausible choices. It's like asking someone to write a review while randomly swapping in words from a dictionary.

### Summary

| Temperature | What It Feels Like | Quality of Output |
|---|---|---|
| 0.2 | Playing it extremely safe | Correct but boring and repetitive |
| 0.5 | Thoughtful and natural | Best balance of quality and variety |
| 1.0 | Creative and expressive | Interesting but sometimes grammatically off |
| 1.5 | Throwing darts at a dictionary | Mostly incoherent, only fragments make sense |

**Bottom line:** For generating realistic wine reviews, a temperature around **0.5** works best, it produces text that sounds natural and varied without sacrificing readability. Going lower makes the output too safe, and going higher quickly leads to gibberish. The choice of temperature ultimately depends on whether we prioritize correctness (low) or creativity (high).

## AI Usage Note

GitHub Copilot was used minimally for:
- **Phrasing observations:** Helped refine how I described the temperature comparison results after I analyzed the generated outputs myself
- **Minor code insights:** Occasional suggestions for syntax and formatting, but all core implementation (GPT architecture, transformer block, token/position embedding, training pipeline, text generation) was written and understood by me

---

## References

Foster, D. (2022). *Generative Deep Learning* (2nd ed.). O'Reilly Media, Inc.